# Analyse de toutes les expériences exportées

Ce notebook recharge chaque expérience contenant `backtest_metrics.csv` dans `exports`, reconstruit les metrics du bassin complet pour les périodes économiques et les fenêtres glissantes, puis imprime uniquement les metrics clés.

In [ ]:
from pathlib import Path
import importlib
import sys
import pandas as pd

PLUGIN_DIR = Path(r"C:\dev\factor_backtest")
if str(PLUGIN_DIR) not in sys.path:
    sys.path.insert(0, str(PLUGIN_DIR))

import func
importlib.reload(func)

from BacktestEngine import (
    PtfBuilder,
    build_periods_from_breakpoints,
    build_rolling_periods,
)
from func import plot_performance_comparison

print("Plugin chargé")

In [ ]:
EXPORT_ROOT = PLUGIN_DIR / "exports"
COMPARISON_MAX_TESTS = 8  # Augmentez cette valeur seulement si chaque figure reste lisible.
PERIOD_BREAKPOINTS = [2009, 2013, 2017, 2020, 2022, 2024, 2026]
HEADLINE_PERIODS = [
    {
        "id": "headline_2009_2019", "label": "2009-2019",
        "start": "2009-01-01", "end": "2019-12-31",
        "robust_score_comparable": True,
    },
    {
        "id": "headline_2020_2023", "label": "2020-2023",
        "start": "2020-01-01", "end": "2023-12-31",
        "robust_score_comparable": True,
    },
    {
        "id": "headline_since_2024", "label": "Depuis 2024",
        "start": "2024-01-01", "end": None,
        "robust_score_comparable": False,
    },
]
RUN_ROLLING_ANALYSIS = False  # Passez à True pour calculer les fenêtres glissantes.
ROLLING_WINDOW_SPECS = [
    {"window_months": 60, "step_months": 12},
    {"window_months": 36, "step_months": 6},
    {"window_months": 12, "step_months": 3},
]
ROLLING_36M_POSITIVE_THRESHOLD = 0.65
PRINT_ROLLING_DETAILS = True
SHOW_PLOTS = True


In [ ]:
key_metric_columns = [
    "selected_for_figure", "display_label",
    "test_path", "test_name", "test_type", "metric",
    "benchmark", "raw_variables", "composition_recipe",
    "period_group", "period_label", "selection_metric",
    "robust_score_comparable",
    "requested_start_date", "requested_end_date",
    "actual_start_date", "actual_end_date",
    "observation_count", "years",
    "robust_score", "robust_score_rank_global", "robust_score_rank_within_type",
    "active_cagr_rank_global", "active_cagr_rank_within_type",
    "top_total_return", "top_annualized_return",
    "top_annualized_volatility", "top_sharpe_ratio",
    "top_max_drawdown", "top_sortino_ratio", "top_beta",
    "top_tracking_error", "top_information_ratio",
    "worst_annualized_return", "bench_annualized_return",
    "top_bench_ratio", "top_worst_ratio", "active_max_drawdown",
    "tracking_error_annualized", "min_rolling_3y_cagr",
    "active_cagr", "top_worst_cagr",
]

rolling_metric_columns = [
    "test_path", "test_name", "test_type", "metric",
    "raw_variables", "composition_recipe",
    "period_group", "period_label",
    "window_months", "step_months",
    "requested_start_date", "requested_end_date",
    "actual_start_date", "actual_end_date",
    "observation_count", "years",
    "active_cagr", "top_worst_cagr",
    "top_annualized_return", "top_information_ratio",
    "top_max_drawdown", "top_tracking_error",
    "robust_score", "robust_score_comparable",
]

def _relative_cagr(portfolio_return, reference_return):
    if pd.isna(portfolio_return) or pd.isna(reference_return):
        return float("nan")
    if 1 + reference_return <= 0:
        return float("nan")
    return (1 + portfolio_return) / (1 + reference_return) - 1


def _reconstruct_metrics_from_performances(experiment_dir, periods):
    saved_metrics = pd.read_csv(experiment_dir / "backtest_metrics.csv")
    if "scope" in saved_metrics.columns:
        metadata_rows = saved_metrics.loc[
            saved_metrics["scope"].astype(str).eq("total")
        ]
    else:
        metadata_rows = saved_metrics
    metadata_by_path = (
        metadata_rows.drop_duplicates("test_path").set_index("test_path").to_dict(
            orient="index"
        )
    )
    sources = func._load_saved_performances(experiment_dir)
    metric_calculator = PtfBuilder.__new__(PtfBuilder)
    total_rows = []
    subperiod_rows = []
    required_columns = ["Top", "Worst", "Bench"]

    for test_path, source in sources.items():
        performance = source["performance"].copy()
        missing_columns = [
            column for column in required_columns if column not in performance.columns
        ]
        if missing_columns:
            raise KeyError(
                f"Performance absente pour {test_path} : {missing_columns}"
            )
        performance = performance.loc[:, required_columns].sort_index()
        performance.index = pd.to_datetime(performance.index, errors="coerce")
        performance = performance.loc[performance.index.notna()].dropna()
        if performance.empty:
            continue

        base = dict(metadata_by_path.get(test_path, {}))
        metadata = source.get("metadata", {})
        fallback_metadata = {
            "test_path": test_path,
            "test_name": source.get("test_name") or test_path,
            "test_type": metadata.get("test_type"),
            "metric": metadata.get("metric"),
        }
        for column, value in fallback_metadata.items():
            if column not in base or pd.isna(base[column]):
                base[column] = value
        base["test_path"] = test_path

        classic_metrics = {
            portfolio: metric_calculator._calculate_classic_metrics(
                performance[portfolio],
                benchmark=performance["Bench"],
            )
            for portfolio in required_columns
        }
        robust_score, top_bench_ratio, top_worst_ratio = (
            metric_calculator._calculate_robust_score(
                performance["Top"],
                performance["Worst"],
                performance["Bench"],
                store_metrics=True,
            )
        )
        robust_metrics = dict(metric_calculator.robust_metrics)
        total_row = {
            **base,
            "scope": "total",
            "period_id": "total",
            "period_label": "Période totale",
            "period_group": "total",
            "window_months": None,
            "step_months": None,
            "robust_score_comparable": True,
            "actual_start_date": performance.index[0].date().isoformat(),
            "actual_end_date": performance.index[-1].date().isoformat(),
            "observation_count": len(performance),
            "years": max((len(performance) - 1) / 252, 0),
            "top_cagr": classic_metrics["Top"]["annualized_return"],
            "worst_cagr": classic_metrics["Worst"]["annualized_return"],
            "bench_cagr": classic_metrics["Bench"]["annualized_return"],
            "active_cagr": _relative_cagr(
                classic_metrics["Top"]["annualized_return"],
                classic_metrics["Bench"]["annualized_return"],
            ),
            "top_worst_cagr": _relative_cagr(
                classic_metrics["Top"]["annualized_return"],
                classic_metrics["Worst"]["annualized_return"],
            ),
            "robust_score": robust_score,
            "top_bench_ratio": top_bench_ratio,
            "top_worst_ratio": top_worst_ratio,
            **robust_metrics,
        }
        for portfolio, metrics in classic_metrics.items():
            total_row.update({
                f"{portfolio.lower()}_{metric}": value
                for metric, value in metrics.items()
            })
        total_rows.append(total_row)

        period_metrics = metric_calculator._calculate_metrics_for_periods(
            performance, periods
        )
        for period_row in period_metrics.to_dict(orient="records"):
            subperiod_rows.append({
                **base,
                **period_row,
                "scope": "subperiod",
                "test_path": test_path,
            })

    reconstructed_metrics = pd.DataFrame([*total_rows, *subperiod_rows])
    return func._finalize_backtest_metrics(reconstructed_metrics)


def _period_definitions(period_breakpoints):
    periods = [
        period
        for period in build_periods_from_breakpoints(period_breakpoints)
        if not period["id"].startswith("before_")
    ]
    for period in periods:
        period["period_group"] = "economic"
        if period["id"].startswith("since_"):
            period["label"] = f"{max(period_breakpoints)} YTD"
        if period.get("start") and period.get("end"):
            duration = pd.Timestamp(period["end"]) - pd.Timestamp(period["start"])
            period["robust_score_comparable"] = duration.days + 1 >= 365 * 3
        else:
            period["robust_score_comparable"] = False
    headline_periods = []
    for period in HEADLINE_PERIODS:
        headline_period = dict(period)
        headline_period["period_group"] = "headline"
        headline_periods.append(headline_period)
    return [
        {
            "id": "total", "label": "Période totale",
            "start": None, "end": None, "period_group": "total",
        },
        *headline_periods,
        *periods,
    ]


def _shared_performance_bounds(experiment_dirs):
    """Retourne les dates communes à tous les tests de toutes les expériences."""
    starts = []
    ends = []
    for experiment_dir in experiment_dirs:
        for source in func._load_saved_performances(experiment_dir).values():
            performance = source["performance"]
            columns = ["Top", "Worst", "Bench"]
            performance = performance.loc[:, columns].dropna()
            if not performance.empty:
                starts.append(pd.Timestamp(performance.index[0]))
                ends.append(pd.Timestamp(performance.index[-1]))
    if not starts:
        raise ValueError("Aucune performance disponible pour les fenêtres glissantes.")
    return max(starts), min(ends)


def _rolling_period_definitions(start_date, end_date):
    """Construit les trois familles de fenêtres glissantes."""
    periods = []
    for specification in ROLLING_WINDOW_SPECS:
        periods.extend(build_rolling_periods(
            start_date,
            end_date,
            window_months=specification["window_months"],
            step_months=specification["step_months"],
        ))
    return periods


def _summarize_rolling_metrics(rolling_metrics):
    """Résume chaque facteur séparément pour chaque longueur de fenêtre."""
    rows = []
    for (test_path, period_group), group in rolling_metrics.groupby(
        ["test_path", "period_group"], sort=True
    ):
        active = pd.to_numeric(group["active_cagr"], errors="coerce").dropna()
        top_worst = pd.to_numeric(
            group["top_worst_cagr"], errors="coerce"
        ).dropna()
        information_ratio = pd.to_numeric(
            group["top_information_ratio"], errors="coerce"
        ).dropna()
        max_drawdown = pd.to_numeric(
            group["top_max_drawdown"], errors="coerce"
        ).dropna()
        tracking_error = pd.to_numeric(
            group["top_tracking_error"], errors="coerce"
        ).dropna()
        first_row = group.iloc[0]
        positive_ratio = (active > 0).mean() if not active.empty else float("nan")
        worst_period_id = (
            group.loc[active.idxmin(), "period_id"]
            if not active.empty else None
        )
        rows.append({
            "test_path": test_path,
            "test_name": first_row.get("test_name"),
            "test_type": first_row.get("test_type"),
            "metric": first_row.get("metric"),
            "period_group": period_group,
            "window_count": len(active),
            "active_positive_ratio": positive_ratio,
            "top_worst_positive_ratio": (
                (top_worst > 0).mean() if not top_worst.empty else float("nan")
            ),
            "min_active_cagr": active.min() if not active.empty else float("nan"),
            "min_top_worst_cagr": (
                top_worst.min() if not top_worst.empty else float("nan")
            ),
            "min_information_ratio": (
                information_ratio.min()
                if not information_ratio.empty else float("nan")
            ),
            "max_top_drawdown": (
                max_drawdown.max() if not max_drawdown.empty else float("nan")
            ),
            "max_tracking_error": (
                tracking_error.max() if not tracking_error.empty else float("nan")
            ),
            "worst_period_id": worst_period_id,
            "passes_36m_threshold": (
                positive_ratio >= ROLLING_36M_POSITIVE_THRESHOLD
                if period_group == "rolling_36m" else None
            ),
        })
    return pd.DataFrame(rows)


def _prepare_reconstructed_comparisons(experiment_dir, metrics, periods):
    comparisons = {}
    for period in periods:
        # Les périodes courtes sont classées par active CAGR.
        selection_metric = (
            "robust_score"
            if period.get("robust_score_comparable", True)
            else "active_cagr"
        )
        selection_metrics = metrics.copy()
        period_rows = selection_metrics["period_id"].astype(str).eq(
            str(period["id"])
        )
        selection_metrics.loc[period_rows, "robust_score"] = selection_metrics.loc[
            period_rows, selection_metric
        ]
        selections, ratio_definitions = func.build_performance_comparison_definitions(
            export_dir=experiment_dir,
            max_tests=COMPARISON_MAX_TESTS,
            period_id=period["id"],
            metrics=selection_metrics,
        )
        performance, composition = func.combine_backtest_performances(
            export_dir=experiment_dir,
            selections=selections,
            return_composition=True,
        )
        ratios = func.calculate_performance_ratios(
            performance,
            benchmark_column="Benchmark",
            ratio_definitions=ratio_definitions,
        )
        comparisons[period["id"]] = {
            "performance": performance,
            "ratios": ratios,
            "composition": composition,
            "performance_selection": selections,
            "ratio_definitions": ratio_definitions,
            "selection_metric": selection_metric,
            "period": period,
            "period_definitions": periods,
        }
    return comparisons

experiment_dirs = sorted(
    experiment_dir
    for experiment_dir in EXPORT_ROOT.iterdir()
    if experiment_dir.is_dir()
    and (experiment_dir / "backtest_metrics.csv").exists()
)
if not experiment_dirs:
    raise FileNotFoundError(
        f"Aucun dossier d'expérience avec backtest_metrics.csv dans {EXPORT_ROOT}"
    )

period_definitions = _period_definitions(PERIOD_BREAKPOINTS)
common_start = None
common_end = None
rolling_period_definitions = []
if RUN_ROLLING_ANALYSIS:
    common_start, common_end = _shared_performance_bounds(experiment_dirs)
    rolling_period_definitions = _rolling_period_definitions(
        common_start, common_end
    )
calculation_periods = [
    *period_definitions[1:],
    *rolling_period_definitions,
]
comparisons_by_period_by_experiment = {}
comparison_figures = {}
metrics_by_experiment = {}
rolling_metrics_by_experiment = {}
rolling_summary_by_experiment = {}

print("\n" + "=" * 100)
print(f"EXPORT_ROOT : {EXPORT_ROOT}")
print(f"PERIOD_BREAKPOINTS : {PERIOD_BREAKPOINTS}")
print("Périodes reconstruites :")
print(", ".join(period["label"] for period in period_definitions))
if RUN_ROLLING_ANALYSIS:
    print(f"Plage commune : {common_start.date()} → {common_end.date()}")
    print("Fenêtres glissantes :")
    for specification in ROLLING_WINDOW_SPECS:
        period_group = f"rolling_{specification['window_months']}m"
        window_count = sum(
            period["period_group"] == period_group
            for period in rolling_period_definitions
        )
        print(
            f"{specification['window_months']} mois / pas "
            f"{specification['step_months']} mois : {window_count} fenêtres"
        )
else:
    print("Analyse des fenêtres glissantes désactivée.")
print(f"Nombre d'expériences : {len(experiment_dirs)}")
print("Metrics clés recalculées à partir des performance CSV")
print("=" * 100)

for experiment_dir in experiment_dirs:
    experiment_name = experiment_dir.name
    print("\n" + "@" * 100)
    print(f"EXPÉRIENCE : {experiment_name}")
    print(f"EXPORT_DIR : {experiment_dir}")

    prompt_metrics = _reconstruct_metrics_from_performances(
        experiment_dir, calculation_periods
    )
    rolling_metrics = prompt_metrics.loc[
        prompt_metrics["period_group"].astype(str).str.startswith("rolling_")
    ].copy()
    rolling_summary = _summarize_rolling_metrics(rolling_metrics)
    metrics_by_experiment[experiment_name] = prompt_metrics
    rolling_metrics_by_experiment[experiment_name] = rolling_metrics
    rolling_summary_by_experiment[experiment_name] = rolling_summary
    comparisons_by_period = _prepare_reconstructed_comparisons(
        experiment_dir, prompt_metrics, period_definitions
    )
    comparisons_by_period_by_experiment[experiment_name] = comparisons_by_period

    experiment_figures = {}
    for period_id, comparison in comparisons_by_period.items():
        experiment_figures[period_id] = plot_performance_comparison(
            performance=comparison["performance"],
            ratios=comparison["ratios"],
            benchmark_column="Benchmark",
            title=f"{experiment_name} | Comparaison des performances",
            save_path=None,
            show_plot=SHOW_PLOTS,
            rebase=True,
            show_worst_performance=False,
            period_definitions=comparison["period_definitions"],
            default_period_id=period_id,
        )
    comparison_figures[experiment_name] = experiment_figures

    for period_id, comparison in comparisons_by_period.items():
        period = comparison["period"]
        selected_top = [
            (label, test_path)
            for label, (test_path, portfolio)
            in comparison["performance_selection"].items()
            if portfolio == "Top"
        ]
        print("\n" + "#" * 100)
        print(f"PERIOD_ID : {period_id}")
        print(f"Période : {period['label']}")
        print(f"Début réel : {period.get('start')}")
        print(f"Fin réelle : {period.get('end')}")
        print(f"Métrique de sélection : {comparison['selection_metric']}")
        selected_paths = [test_path for _, test_path in selected_top]
        display_labels = {label_path: label for label, label_path in selected_top}
        period_metrics = prompt_metrics.loc[
            prompt_metrics["period_id"].astype(str).eq(str(period_id))
        ].copy()
        if not period_metrics.empty:
            period_metrics["selection_metric"] = comparison[
                "selection_metric"
            ]
            period_metrics["selected_for_figure"] = period_metrics[
                "test_path"
            ].isin(selected_paths)
            period_metrics["_selection_order"] = period_metrics["test_path"].map(
                {test_path: index for index, test_path in enumerate(selected_paths)}
            ).fillna(len(selected_paths))
            period_metrics["display_label"] = period_metrics["test_path"].map(
                display_labels
            )
            period_metrics = period_metrics.sort_values("_selection_order").drop(
                columns="_selection_order",
            )
        available_metric_columns = [
            column for column in key_metric_columns if column in period_metrics.columns
        ]
        print("Bassin complet et metrics clés (CSV) :")
        print(period_metrics[available_metric_columns].to_csv(index=False))

    if RUN_ROLLING_ANALYSIS:
        print("\n" + "%" * 100)
        print("Synthèse simple des fenêtres glissantes (CSV) :")
        print(rolling_summary.to_csv(index=False))

    if RUN_ROLLING_ANALYSIS and PRINT_ROLLING_DETAILS:
        available_rolling_columns = [
            column
            for column in rolling_metric_columns
            if column in rolling_metrics.columns
        ]
        rolling_metrics = rolling_metrics.sort_values(
            ["period_group", "requested_start_date", "test_path"]
        )
        print("\n" + "&" * 100)
        print("Toutes les fenêtres glissantes et metrics clés (CSV) :")
        print(rolling_metrics[available_rolling_columns].to_csv(index=False))